In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
BASE = "/content/drive/MyDrive/T2_Project_Tanmay"
folders = [
    f"{BASE}/Notebooks",
    f"{BASE}/Data",
    f"{BASE}/Models",
    f"{BASE}/Visualizations",
    f"{BASE}/Report",
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("Created folders under:", BASE)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Loading dataset

df = pd.read_csv("german_credit_data.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Count plot of target variable(Risk)

from numpy import place
plt.figure(figsize=(6,4))
sns.countplot(x='Risk', data=df)
plt.title("Count of Risk")
plt.show()

In [ ]:
# Histograms

df[['Age', 'Credit amount', 'Duration']].hist(figsize=(10,5))
plt.show()

In [ ]:
# Box plots

plt.figure(figsize=(6,4))
sns.boxplot(x='Risk', y='Credit amount', data=df)
plt.show()

In [ ]:
# Coorelation Heatmap

df.numeric = df.select_dtypes(include=np.number)

plt.figure(figsize=(10,6))
sns.heatmap(df.numeric.corr(), annot=True)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
df['Monthly_Burden'] = df['Credit amount'] / df['Duration']
sns.kdeplot(x='Monthly_Burden', hue='Risk', data=df, fill= True)
plt.title("Monthly Burden Distribution by Risk")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Job', hue='Risk', data=df)
plt.title("Risk Distribution Across Job Types")
plt.show()

In [ ]:
# DATA PREPROCESSING

In [ ]:
# Handling missing values

df['Saving accounts'] = df['Saving accounts'].fillna('no_account')
df['Checking account'] = df['Checking account'].fillna('no_account')

In [ ]:
df.isnull().sum()

In [ ]:
# Target Encoding

df['Risk'] = df['Risk'].map({'good': 0, 'bad': 1})
df['Risk'].value_counts()

In [ ]:
df.select_dtypes(include='object').columns

In [ ]:
# Applying one-hot encoding to categorical variables

df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
# Seperating features and target

X = df.drop('Risk', axis=1)
y = df['Risk']

In [ ]:
# train test split

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

In [ ]:
X_train.to_csv(f"{BASE}/Data/X_train.csv", index=False)
X_test.to_csv(f"{BASE}/Data/X_test.csv", index=False)
print("Saved:", f"{BASE}/Data/X_train.csv and X_test.csv")

In [ ]:
# Scaling

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 1.4 Feature Engineering

In [ ]:
# Credit_to_Duration_Ratio: it measures the monthly repayment pressure of the borrower. A high value indicates larger loan burden per month, which may increase the probability of default.

df['Credit_to_Duration_Ratio'] = df['Credit amount'] / df['Duration']


In [ ]:
# Age_Group: Categorises borrowers into life-stage segments. Younger borrowers may have unstable income and higher risk, while older ones tend to be financially stable.

df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 50, 100], labels=['18-25', '26-35', '36-50', '50+'])

In [ ]:
# High_Risk_Purpose: binary feature that flags loan purposes historically associated with higher financial uncertainity.
#                     Certain loan purposes may indicate to higher risk default due to car loans, education loans etc.

High_Risk = ['car', 'education','repairs' ]
df['High_Risk_Purpose'] = (df[['Purpose_car', 'Purpose_education', 'Purpose_repairs']].sum(axis=1) > 0).astype(int)

In [ ]:
# Account_Stability: captures the financial strength of a borrower by combining savings and checking account categories.

df['Account_Stability'] = (df[['Saving accounts_rich', 'Saving accounts_quite rich', 'Checking account_rich']].sum(axis=1) > 0).astype(int)